## Load data from hf

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from datasets import load_dataset

In [ ]:
os.getenv("HF_TOKEN")

In [ ]:
df = load_dataset("viamr-project/reasoning-frame-hint", token=os.getenv("HF_TOKEN"))
df

In [ ]:
sample = df["train"][0]

# Hint

In [ ]:
from services.amr_hint.amr_hint import AMRHint

amr = """
(w / want-01
    :ARG0 (b / boy)
    :ARG1 (g / go-01
              :ARG0 b))
""".strip()

hint_gen = AMRHint()
hints = hint_gen.get_hints_json(amr)
print(hints)

In [ ]:
from services.amr_hint.prompt_builder import build_prompt

print(build_prompt(sample["sentence"], sample["amr"]))

In [ ]:
partial = hint_gen.get_hints_partial(amr, percentage=0.3)
print(partial)

In [ ]:
import json

# JSON (string) - phù hợp để đưa thẳng vào model
print(hint_gen.get_hints_json(amr, indent=2))

# Hoặc lấy dict Python (structured object)
structured = hint_gen.get_hints_structured(amr)
structured

# vLLM

In [ ]:
from vllm import LLM, SamplingParams

In [ ]:
import os
from vllm import SamplingParams

vllm_config = {
    "model": "Qwen/Qwen3.6-35B-A3B-FP8",
    "tensor_parallel_size": 1,
    "gpu_memory_utilization": 0.85,
    "max_model_len": 4096,
    
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 1.5,
    "repetition_penalty": 1.0,
    "max_tokens": 1024,
}

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

sampling_params = SamplingParams(
    temperature=vllm_config["temperature"],
    top_p=vllm_config["top_p"],
    top_k=vllm_config["top_k"],
    min_p=vllm_config["min_p"],
    presence_penalty=vllm_config["presence_penalty"],
    repetition_penalty=vllm_config["repetition_penalty"],
    max_tokens=vllm_config["max_tokens"],
)

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name()}")


try:
    vllm_model = LLM(
        model=vllm_config["model"],
        tensor_parallel_size=vllm_config["tensor_parallel_size"],
        gpu_memory_utilization=vllm_config["gpu_memory_utilization"],
        max_model_len=vllm_config["max_model_len"],
        hf_token=os.getenv("HF_TOKEN")
    )
    print(f"✓ vLLM model loaded successfully: {vllm_config["model"]}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
from vllm import LLM

prompt = "AMR trong NLP là gì?"

outputs = vllm_model.generate(prompt, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

In [ ]:
print(generated_text)

In [ ]:
sample = {"amr":"(n / need-01\n   :ARG0 (h / he)\n   :ARG1 (c / conduct-01\n            :ARG0 h\n            :ARG1 (m / meet-03\n                     :ARG0 (p / person\n                              :name (n2 / name\n                                        :op1 \"Palin\"))\n                     :ARG1 (p2 / publication\n                               :name (n3 / name\n                                         :op1 \"NBC\"))\n                     :mod (s / summit\n                             :mod (b / beer)))))","sentence":"He needs to conduct a beer summit between Palin and NBC.","model_respose":["Let me analyze this carefully.\n\nThe sentence is: \"He needs to conduct a beer summit between Palin and NBC.\"\n\nKey elements:\n1. \"He\" - a person (ARG0 of need-01)\n2. \"needs\" - main predicate `need-01`\n3. \"to conduct a beer summit\" - ARG1 of need-01, with `conduct-01` as sub-predicate\n4. \"beer summit\" - ARG1 of conduct-01\n5. \"between Palin and NBC\" - this is a `meet-03` event, with Palin and NBC as the parties\n\nLet me trace through:\n\nRoot: `need-01`\n- ARG0: `person` (he)\n- ARG1: `conduct-01`\n  - ARG0: `person` (same \"he\")\n  - ARG1: `meet-03` (the \"beer summit\" is a meeting)\n    - ARG0: `person` \"Palin\"\n    - ARG1: `organization` \"NBC\"\n\nNow, \"beer summit\" modifies the meet-03 event - it describes the type of meeting (a summit involving beer). I'll treat \"beer\" as a modifier on the meeting domain.\n\nLet me also consider that \"conduct\" might take the summit as a whole event rather than just the participants. So `conduct-01`'s ARG1 is the `meet-03` event.\n\nFor names: \"Palin\" and \"NBC\" are named entities. \"Palin\" is a person, \"NBC\" is an organization.\n\nLet me build the AMR:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (m / meet-03\n                     :ARG0 (p2 / person :name (n2 / name :op1 \"Palin\"))\n                     :ARG1 (o / organization :name (n3 / name :op1 \"NBC\"))\n                     :mod (b / beer))))\n```\n\nWait, let me reconsider. \"Beer summit\" is a named event concept - it's a specific type of meeting. Maybe \"summit\" is the head noun and \"beer\" is a modifier. But in AMR, \"summit\" as a meeting would likely map to `meet-03` or similar.\n\nActually, looking at the hints more carefully:\n- The frames given are: need-01, conduct-01, meet-03\n- meet-03: \"get together (with)\", ARG0 and ARG1 are the two parties\n\nSo \"a beer summit between Palin and NBC\" maps to a `meet-03` event with Palin and NBC as the two parties, and \"beer\" as a modifier.\n\nLet me produce the AMR:\n\n```\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (m / meet-03\n                     :ARG0 (p2 / person :name (n2 / name :op1 \"Palin\"))\n                     :ARG1 (o / organization :name (n3 / name :op1 \"NBC\"))\n                     :mod (b / beer))))\n```\n\nThis seems right. The \"he\" is the same person conducting the meeting, so I reused the person variable `p` for conduct-01's ARG0.<think>\nThe sentence structure is:\n- \"He needs to conduct a beer summit between Palin and NBC.\"\n- Main predicate: need-01, with \"He\" as ARG0 (the needer) and the infinitive clause as ARG1 (thing needed).\n- The infinitive \"to conduct a beer summit between Palin and NBC\" uses conduct-01.\n  - conduct-01's ARG0 is the same \"he\" (reusing the person variable).\n  - conduct-01's ARG1 is \"a beer summit between Palin and NBC\".\n- \"summit\" here is a meeting event → meet-03.\n  - meet-03's ARG0 = Palin (person)\n  - meet-03's ARG1 = NBC (organization)\n  - \"beer\" is a modifier on the meet-03 event (type of summit).\n- Names are represented with :name and :op1.\n</think>\n\n<amr>\n(n / need-01\n   :ARG0 (p / person)\n   :ARG1 (c / conduct-01\n            :ARG0 p\n            :ARG1 (m / meet-03\n                     :ARG0 (p2 / person :name (n2 / name :op1 \"Palin\"))\n                     :ARG1 (o / organization :name (n3 / name :op1 \"NBC\"))\n                     :mod (b / beer))))\n</amr>"]}


sample["model_respose"][0]

In [ ]:
print(sample["model_respose"][0])

In [ ]:
import json

with open("data/systhetic_data.jsonl", "r") as f:
    data = [json.loads(line) for line in f if line.strip()]

In [ ]:
data[0]

In [ ]:
import json
from pathlib import Path

output_path = Path("data/systhetic_data.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

output_path

## Đánh giá độ chính xác với smatchpp

In [ ]:
import json
import re
from smatchpp import Smatchpp

# ── helpers ────────────────────────────────────────────────────────────────────

def extract_amr_from_response(model_response: list[str] | str) -> str | None:
    """
    Extract AMR string from <amr>...</amr> tags in model_response.
    model_response can be a list of strings or a plain string.
    Returns the first match found, or None.
    """
    text = "\n".join(model_response) if isinstance(model_response, list) else model_response
    match = re.search(r"<amr>(.*?)</amr>", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


def score_amr_pair(gold: str, pred: str, scorer: Smatchpp) -> dict:
    """
    Score a single (gold, pred) AMR pair.
    Returns {'F1': float, 'Precision': float, 'Recall': float}
    or {'F1': 0.0, 'Precision': 0.0, 'Recall': 0.0} on parse errors.
    """
    try:
        result = scorer.score_pair(gold, pred)
        return result["main"]          # {'F1': ..., 'Precision': ..., 'Recall': ...}
    except Exception as e:
        return {"F1": 0.0, "Precision": 0.0, "Recall": 0.0, "error": str(e)}


# ── main ───────────────────────────────────────────────────────────────────────

def evaluate(data_path: str = "data/systhetic_data.jsonl") -> None:
    with open(data_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    scorer = Smatchpp()

    results = []
    skipped = 0

    for i, item in enumerate(data):
        gold_amr       = item.get("amr", "").strip()
        sentence       = item.get("sentence", "")
        model_response = item.get("model_respose", item.get("model_response", ""))  # typo-tolerant

        pred_amr = extract_amr_from_response(model_response)

        if not gold_amr or not pred_amr:
            print(f"[{i}] SKIP — missing {'gold' if not gold_amr else 'pred'} AMR | sentence: {sentence!r}")
            skipped += 1
            continue

        scores = score_amr_pair(gold_amr, pred_amr, scorer)

        results.append({
            "index":     i,
            "sentence":  sentence,
            "gold_amr":  gold_amr,
            "pred_amr":  pred_amr,
            **scores,
        })

        print(
            f"[{i:>4}] F1={scores['F1']:5.1f}  "
            f"P={scores['Precision']:5.1f}  "
            f"R={scores['Recall']:5.1f}  | {sentence[:70]}"
        )

    # ── aggregate ──────────────────────────────────────────────────────────────
    if results:
        avg_f1   = sum(r["F1"]        for r in results) / len(results)
        avg_prec = sum(r["Precision"] for r in results) / len(results)
        avg_rec  = sum(r["Recall"]    for r in results) / len(results)

        print("\n" + "=" * 70)
        print(f"  Evaluated : {len(results)} samples  (skipped: {skipped})")
        print(f"  Avg F1    : {avg_f1:.2f}")
        print(f"  Avg Prec  : {avg_prec:.2f}")
        print(f"  Avg Recall: {avg_rec:.2f}")
        print("=" * 70)
    else:
        print("No valid pairs to evaluate.")

    # ── save detailed results ──────────────────────────────────────────────────
    out_path = "amr_eval_results.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            # convert numpy floats → plain Python floats for JSON
            row = {k: (float(v) if hasattr(v, "item") else v) for k, v in r.items()}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"\nDetailed results saved to: {out_path}")


if __name__ == "__main__":
    import sys
    path = sys.argv[1] if len(sys.argv) > 1 else "data/systhetic_data.jsonl"
    evaluate(path)

In [ ]:
import json
import re
from smatchpp import Smatchpp

# ── helpers ────────────────────────────────────────────────────────────────────

def extract_amr_from_response(model_response: list[str] | str) -> str | None:
    """
    Extract AMR string from <amr>...</amr> tags in model_response.
    model_response can be a list of strings or a plain string.
    Returns the first match found, or None.
    """
    text = "\n".join(model_response) if isinstance(model_response, list) else model_response
    match = re.search(r"<amr>(.*?)</amr>", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


def score_amr_pair(gold: str, pred: str, scorer: Smatchpp) -> dict:
    """
    Score a single (gold, pred) AMR pair.
    Returns {'F1': float, 'Precision': float, 'Recall': float}
    or {'F1': 0.0, 'Precision': 0.0, 'Recall': 0.0} on parse errors.
    """
    try:
        result = scorer.score_pair(gold, pred)
        return result["main"]          # {'F1': ..., 'Precision': ..., 'Recall': ...}
    except Exception as e:
        return {"F1": 0.0, "Precision": 0.0, "Recall": 0.0, "error": str(e)}


# ── main ───────────────────────────────────────────────────────────────────────

def evaluate(data_path: str = "data/systhetic_data.jsonl") -> None:
    with open(data_path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    scorer = Smatchpp()

    results = []
    skipped = 0

    for i, item in enumerate(data):
        gold_amr       = item.get("amr", "").strip()
        sentence       = item.get("sentence", "")
        model_response = item.get("model_respose", item.get("model_response", ""))  # typo-tolerant

        pred_amr = extract_amr_from_response(model_response)

        if not gold_amr or not pred_amr:
            print(f"[{i}] SKIP — missing {'gold' if not gold_amr else 'pred'} AMR | sentence: {sentence!r}")
            skipped += 1
            continue

        scores = score_amr_pair(gold_amr, pred_amr, scorer)

        results.append({
            "index":     i,
            "sentence":  sentence,
            "gold_amr":  gold_amr,
            "pred_amr":  pred_amr,
            **scores,
        })

        print(
            f"[{i:>4}] F1={scores['F1']:5.1f}  "
            f"P={scores['Precision']:5.1f}  "
            f"R={scores['Recall']:5.1f}  | {sentence[:70]}"
        )

    # ── aggregate ──────────────────────────────────────────────────────────────
    if results:
        avg_f1   = sum(r["F1"]        for r in results) / len(results)
        avg_prec = sum(r["Precision"] for r in results) / len(results)
        avg_rec  = sum(r["Recall"]    for r in results) / len(results)

        print("\n" + "=" * 70)
        print(f"  Evaluated : {len(results)} samples  (skipped: {skipped})")
        print(f"  Avg F1    : {avg_f1:.2f}")
        print(f"  Avg Prec  : {avg_prec:.2f}")
        print(f"  Avg Recall: {avg_rec:.2f}")
        print("=" * 70)
    else:
        print("No valid pairs to evaluate.")

    # ── save detailed results ──────────────────────────────────────────────────
    out_path = "amr_eval_results.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for r in results:
            # convert numpy floats → plain Python floats for JSON
            row = {k: (float(v) if hasattr(v, "item") else v) for k, v in r.items()}
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"\nDetailed results saved to: {out_path}")


if __name__ == "__main__":
    evaluate("data/systhetic_data.jsonl")